## Creation of xyz/cif files for different sizes and for the predefined Wulff forms from a dataset of compounds (cif files)

### Imports

In [6]:
import os
import sys
import tkinter

print(os.getcwd())
cwd0 = './styles/'
sys.path.append(cwd0)
from pathlib import Path
import re
import numpy as np
import pandas as pd
import visualID as vID
from visualID import  fg, hl, bg


import ase
from ase.atoms import Atoms
from ase.geometry import cellpar_to_cell
from ase.spacegroup import get_spacegroup
from ase import io
from ase.io import read
from ase.io import write
from pyNanoMatBuilder import crystalNPs as cyNP
from pyNanoMatBuilder import utils as pyNMBu
from pyNanoMatBuilder import data
import importlib


from ase.io import write
from ase.visualize import view


from pyNanoMatBuilder import crystalNPs as cyNP
from pyNanoMatBuilder import utils as pyNMBu
from pyNanoMatBuilder import data
import importlib



/home/sara/pyNanoMatBuilder


### Definition of the PredifinedWulffFiles class : functions that load cif files, extract the data, and call the crystalNPs class to create the NPs if the lattice matches with the Wulff form

In [7]:
class PredifinedWulffFiles :
    
    def __init__(self, cif_data, wulff_shapes,sizes,form: str=None):
        """
        Initialize the class with CIF data, Wulff shapes information and size.
        """
        self.cif_data = cif_data  # Dictionnaire contenant les fichiers CIF
        self.wulff_shapes = wulff_shapes  # DataFrame des formes de Wulff
        self.loaded_cifs = {}  # Stocker les fichiers CIF chargés
        self.sizes= sizes
        self.form= form

    
    def get_crystal_type(self):
        
        spacegroup_number = self.ucSG.no  # Numéro du groupe d'espace

        # système cristallin à partir du numéro du groupe d'espace
        if 195 <= spacegroup_number <= 230:  # Cubic
            if spacegroup_number == 225:
                return 'fcc'
            elif spacegroup_number == 229:
                return 'bcc'
            else:
                return 'other cubic'
        elif 168 <= spacegroup_number <= 194:  # Hexagonal
            return 'hcp'
        elif 75 <= spacegroup_number <= 142:  # Tetragonal
            return 'tetragonal'
        elif 16 <= spacegroup_number <= 74:  # Orthorhombic
            return 'orthorhombic'
        elif 3 <= spacegroup_number <= 15:  # Monoclinic
            return 'monoclinic'
        elif 1 <= spacegroup_number <= 2:  # Triclinic
            return 'triclinic'
        else:
            return 'unknown'



    def extract_cif_info(self,cif_file):
        """
        Extract useful information from a CIF file, including:
        - Structure loaded with ASE.
        - Bravais lattice system.
        - Chemical name(s).
        """   
        structure = read(cif_file) # load structure with ase
        self.ucUnitcell = self.cif.cell.cellpar()
        self.ucV = cellpar_to_cell(self.ucUnitcell)
        self.ucBL = self.cif.cell.get_bravais_lattice()
        self.ucSG = get_spacegroup(self.cif,symprec= float(1e-2))
        self.ucFormula = self.cif.get_chemical_formula()
        self.crystal_type = self.get_crystal_type()
        return {
            'structure': self.ucUnitcell,
            'lattice_system':self.ucBL,
            'crystal_name': self.ucFormula,
            'cif_path': cif_file,
            'crystal_type' : self.crystal_type
            
        }
        
    def load_cif(self, cif_file):
        """
        Load a CIF file.
        """      
        cif_folder = "cif_database"
        path2cif = Path(os.path.join(cif_folder, cif_file)).resolve()
        self.cif = io.read(path2cif)
        print("Absolute path to CIF:", path2cif)
        if not path2cif.exists():
            raise FileNotFoundError(f"File {cif_file} not found.")
        if path2cif not in self.loaded_cifs:
            # Utiliser la fonction extract_cif_info pour récupérer les infos
            self.loaded_cifs[path2cif] = self.extract_cif_info(path2cif)
        return self.loaded_cifs[path2cif]

    
    def create_wulff_shapes(self):
        """
        Generate Wulff shapes for all valid combinations of CIF files and Wulff forms.
        """
        for cif_name, cif_file in self.cif_data['cif file'].items():
            # Charger les informations CIF
            self.cif_name=cif_name
            print(f'{cif_name} \033[1m')
            cif_info = self.load_cif(cif_file)
            # Vérifier les formes Wulff compatibles
            if self.form == None :
                for form, row in self.wulff_shapes.iterrows():
                    lattices = [l.strip() for l in row['Bravais lattice'].split(',')]
                    if self.crystal_type  in lattices:
                        
                        index=0 
                        print(f" {bg.LIGHTGREENB} {self.crystal_type} corresponds to the lattices {row['Bravais lattice']} of the Wulff {form} form.\033[0m")
                        # instance of the crystal class for each cif file and possible form and size
                        for i in sizes :
                            index += 1 
                            TestNP = cyNP.Crystal(
                                crystal=f'{cif_name}',
                                userDefCif=cif_info['cif_path'],
                                shape=f"Wulff: {form}",
                                sizesWulff= i,
                                threshold=0.001,
                                thresholdCoreSurface=2,
                                postAnalyzis=True,
                                jmolCrystalShape=True,
                                noOutput=True,
                                aseView=False,
                                skipSymmetryAnalyzis=True
                            ) 
     
                            pyNMBu.writexyz_generalized_crystals(self,path,TestNP,index)
                           
                        
                    else:
                        print(f" {bg.LIGHTREDB} {self.crystal_type} does not correspond to the lattices {row['Bravais lattice']} of the Wulff {form} form .\033[0m")
            
           if not self.form == None :
            for form, row in self.wulff_shapes.iterrows():
                lattices = [l.strip() for l in row['Bravais lattice'].split(',')]
                if self.crystal_type  in lattices:
                    
                    index=0 
                    print(f" {bg.LIGHTGREENB} {self.crystal_type} corresponds to the lattices {row['Bravais lattice']} of the Wulff {form} form.\033[0m")
                    # instance of the crystal class for each cif file and possible form and size
                    for i in sizes :
                        index += 1 
                        TestNP = cyNP.Crystal(
                            crystal=f'{cif_name}',
                            userDefCif=cif_info['cif_path'],
                            shape=f"Wulff: {self.form}",
                            sizesWulff= i,
                            threshold=0.001,
                            thresholdCoreSurface=2,
                            postAnalyzis=True,
                            jmolCrystalShape=True,
                            noOutput=True,
                            aseView=False,
                            skipSymmetryAnalyzis=True
                        ) 
 
                        pyNMBu.writexyz_generalized_crystals(self,path,TestNP,index)
                       
                    
                else:
                    print(f" {bg.LIGHTREDB} {self.crystal_type} does not correspond to the lattices {row['Bravais lattice']} of the Wulff {form} form .\033[0m")
                 
    
    


### Example of use of the class PredifinedWulffFiles that creates all the files

#### modify the tetragonal structure possibilities of shapes (cube possible, ect)

  <span style="color:#008000"> in green : the allowed shapes created (the lattice matches with the shape) </span> \
  <span style="color:#FF0000">in red : the shapes not created (the lattice doesn't match with the shape)</span>

In [8]:
t = pyNMBu.timer()
t.chrono_start()


path='/home/sara/pyNanoMatBuilder/xyz_files_predef_wulff'

sizes=[[1]]

class_test=PredifinedWulffFiles(cif_data=data.pyNMBcif.CIFdf, wulff_shapes=data.WulffShapes.WSdf,sizes=sizes)
class_test.create_wulff_shapes()

print(t.chrono_stop(hdelay=True))

NaCl 
Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1000041-NaCl.cif
  fcc corresponds to the lattices bcc, fcc of the Wulff cube form.
Crystal = 'Sodium chloride' Wulff: cube
instance_class_crystals.crystal 'Sodium chloride'
xyz file created: /home/sara/pyNanoMatBuilder/xyz_files_predef_wulff/NaCl_fcc_cube_0000001_0000000.xyz
cif file created: /home/sara/pyNanoMatBuilder/xyz_files_predef_wulff/NaCl_fcc_cube_0000001_0000000.cif
  fcc corresponds to the lattices fcc of the Wulff trcube form.
Crystal = 'Sodium chloride' Wulff: trcube
instance_class_crystals.crystal 'Sodium chloride'
xyz file created: /home/sara/pyNanoMatBuilder/xyz_files_predef_wulff/NaCl_fcc_trcube_0000001_0000000.xyz
cif file created: /home/sara/pyNanoMatBuilder/xyz_files_predef_wulff/NaCl_fcc_trcube_0000001_0000000.cif
  fcc corresponds to the lattices fcc of the Wulff cubo form.
Crystal = 'Sodium chloride' Wulff: cubo
instance_class_crystals.crystal 'Sodium chloride'
xyz file created: /home/sara/p